# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [9]:
%load_ext dotenv
%dotenv ../05_src/.secrets

The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv


In [10]:
import sys
import sys
import os

sys.path.append('../05_src')
# Get the notebook's parent directory and navigate to 05_src
notebook_dir = os.getcwd()
src_path = os.path.abspath(os.path.join(notebook_dir, '../05_src'))

if src_path not in sys.path:
    sys.path.insert(0, src_path)

print(f"Added to path: {src_path}")
print(f"Path exists: {os.path.exists(src_path)}")

Added to path: c:\DSI\deploying-ai\05_src
Path exists: True


## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [11]:
from langchain_community.document_loaders import PyPDFLoader

file_path = "../02_activities/documents/ai_report_2025.pdf"
loader = PyPDFLoader(file_path)
docs = loader.load()

document_text = ""
for page in docs:
    document_text += page.page_content + "\n"

print(len(docs))
print(document_text)
print(f"Total characters: {len(document_text)}")
print("\n--- First 500 characters (preview) ---")
print(document_text[:500])

26
pg. 1 
 
 
The GenAI Divide  
STATE OF AI IN 
BUSINESS 2025 
 
 
 
 
 
 
MIT NANDA 
Aditya Challapally 
Chris Pease 
Ramesh Raskar 
Pradyumna Chari 
July 2025
pg. 2 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
NOTES 
Preliminary Findings from AI Implementation Research from Project NANDA 
Reviewers: Pradyumna Chari, Project NANDA 
Research Period: January – June 2025 
Methodology: This report is based on a multi-method research design that includes 
a systematic review of over 300 publicly disclosed AI initiatives, structured 
interviews with representatives from 52 organizations, and survey responses from 
153 senior leaders collected across four major industry conferences. 
 Disclaimer: The views expressed in this report are solely those of the authors and 
reviewers and do not reflect the positions of any affiliated employers. 
 Confidentiality Note: All company-specific data and quotes have been 
anonymized to maintain compliance with corporate disclosure policies and 
confidentiality agr

## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
    - Summary: a concise and succinct summary no longer 
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [12]:
import os
from openai import OpenAI
client = OpenAI(base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1', 
                api_key='any value',
                default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')})

response = client.responses.create(temperature=1.5,
    model = 'gpt-4o-mini',
    input = 'Hello world!'
    
)

print(response.output_text)

Hello! How can I assist you today?


In [13]:
import os
from pydantic import BaseModel

class ArticleSummary(BaseModel):
    Author: str
    Title: str
    Relevance: str
    Tone: str
    InputTokens: int
    OutputTokens: int
    Summary: str



In [14]:
selected_tone = "Formal Academic Writing"

In [15]:
system_prompt = """You are a scholarly research assistant specialising in technology and organisational studies.
Your task is to produce a structured summary of a provided article.

Requirements:
- Write in a style of Formal Academic Writing: use precise, discipline-appropriate vocabulary,
  third-person voice, hedged claims where appropriate, and complex but clear sentence structures.
- The Summary field must be concise and no longer than 1000 tokens.
- The Relevance field must be a single paragraph explaining why the article is relevant
  to AI professionals in their professional development.
- Populate Author and Title from the document itself; if the author is not named, write "Not specified".
- Set the Tone field to the exact style name provided by the user.
- Do NOT include token counts in your response — those will be injected programmatically.
- Respond ONLY with the structured JSON matching the schema. Do not add preamble or commentary."""

# ── User prompt (context injected dynamically) ─────────────────────────────────
# 'context' holds the document text; 'selected_tone' is set above.
# We truncate to ~12000 chars to stay well within the model's context window.
context = document_text[:12000]

In [16]:
user_prompt = f"""Please summarise the following article using the tone: {selected_tone}.

Article content:
{context}
"""

print("Prompts constructed. Calling API...")

Prompts constructed. Calling API...


In [17]:
response = client.beta.chat.completions.parse(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user",   "content": user_prompt},
    ],
    response_format=ArticleSummary,
)

In [18]:
summary_obj: ArticleSummary = response.choices[0].message.parsed
summary_obj.InputTokens  = response.usage.prompt_tokens
summary_obj.OutputTokens = response.usage.completion_tokens

print("API call successful.")
print(f"Input tokens : {summary_obj.InputTokens}")
print(f"Output tokens: {summary_obj.OutputTokens}")

API call successful.
Input tokens : 2758
Output tokens: 449


In [19]:
from IPython.display import display, Markdown

display(Markdown(f"""
## Summary Output

**Title:** {summary_obj.Title}  
**Author:** {summary_obj.Author}  
**Tone:** {summary_obj.Tone}  
**Input Tokens:** {summary_obj.InputTokens}  
**Output Tokens:** {summary_obj.OutputTokens}  

### Relevance
{summary_obj.Relevance}

### Summary
{summary_obj.Summary}
"""))


## Summary Output

**Title:** The GenAI Divide: State of AI in Business 2025  
**Author:** Aditya Challapally, Chris Pease, Ramesh Raskar, Pradyumna Chari  
**Tone:** Formal Academic Writing  
**Input Tokens:** 2758  
**Output Tokens:** 449  

### Relevance
This article presents crucial insights into the current landscape of Generative AI in business, particularly highlighting discrepancies between adoption rates and actual transformative impacts. For AI professionals, understanding the barriers to successful implementation and the distinction between pilot projects and scalable solutions is vital. The findings emphasize the importance of process-specific customization and the evaluation of AI tools based on business outcomes, rather than superficial metrics. This knowledge is essential for AI practitioners aiming to enhance their professional development and effectiveness in deploying AI solutions that align with organizational goals.

### Summary
The report titled "The GenAI Divide: State of AI in Business 2025" reveals significant findings regarding the implementation of Generative AI (GenAI) within organizations. Despite substantial investments of $30–40 billion, a staggering 95% of entities reportedly receive no return from their AI initiatives, manifesting a stark divide termed the 'GenAI Divide.' This divide is characterized by the fact that while 80% of organizations have explored widely adopted tools such as ChatGPT and Copilot, these tools mainly enhance individual productivity rather than driving overall performance impact. Additionally, enterprise-grade AI systems face reluctance in adoption due to complexities exceeding expectations, resulting in low conversion rates from pilot to production stages. Insights from structured interviews and a systematic examination of over 300 public AI initiatives indicate that a predominant number of organizations remain on the wrong side of the GenAI Divide, exhibiting high pilot activity but minimal structural transformation across sectors. Key barriers to success include inadequate learning capabilities of GenAI systems, which fail to retain contextual feedback and adapt over time. The report identifies distinct patterns among successful users, suggesting that companies achieving meaningful transformation prioritize process-specific customization and foster external partnerships, contrasting with the limited success observed in larger enterprises, which tend to lag in scaling AI solutions despite leading in pilot initiatives. Overall, the findings underscore an urgent need for organizations to bridge the GenAI Divide by focusing on actionable learning systems that integrate seamlessly with existing business processes.


# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

In [20]:

file_path = "../02_activities/documents/ai_report_2025.pdf"
loader = PyPDFLoader(file_path)
docs = loader.load()

document_text = ""
for page in docs:
    document_text += page.page_content + "\n"

In [21]:
instructions = "ou are a scholarly research assistant specialising in technology and organisational studies.Your task is to produce a structured summary of a provided article."
PROMPT = """
    Summarize the following article in at most four paragraphs. Please include all key characters and plot points.
    <article>
    {article}
    </article>
    In addition to the summary, add an introduction paragraph where you greet the reader and a conclusion where you share an opinion about the story.
"""

In [22]:
import os
client = OpenAI(base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1', 
                api_key='any value',
                default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')})
response = client.responses.create(
    model="gpt-4o-mini",
    instructions=instructions,
    input=[
        {"role": "user", 
         "content": PROMPT.format(article=document_text)}
    ],
    temperature=1.2
)

In [23]:
response.output_text

'### Introduction\nHello! In a rapidly evolving technological landscape, the integration of Generative AI (GenAI) into business practices has sparked substantial interest and investment. The article "The GenAI Divide: State of AI in Business 2025," authored by Aditya Challapally and others from MIT\'s Project NANDA, examines the stark disparity between high adoption rates of GenAI tools and their actual effectiveness in driving meaningful business transformation. It provides insights into the challenges organizations face as they attempt to navigate this divide between potential and performance.\n\n### Summary\nThe article opens with the revelation that, despite a staggering $30-40 billion investment in enterprise GenAI initiatives, an overwhelming 95% of organizations report deriving zero return on investment. Known as the GenAI Divide, this phenomenon highlights a distressing contrast between the high adoption rates of consumer-grade tools like ChatGPT and the failure of custom enter

In [24]:
from deepeval import evaluate
from deepeval.metrics import AnswerRelevancyMetric
from deepeval.test_case import LLMTestCase
from deepeval.models import GPTModel

model = GPTModel(
    model="gpt-4o-mini",
    temperature=0,
    # api_key='any value',
    default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')},
    base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1',
)

metric = AnswerRelevancyMetric(
    threshold=0.7,
    include_reason=True,
    model=model,
    
)

test_case = LLMTestCase(
    input=PROMPT.format(article=document_text),
    actual_output=response.output_text,
    
)

In [25]:
metric.measure(test_case)

c:\DSI\deploying-ai-env\Lib\site-packages\rich\live.py:260: UserWarning: install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

1.0

In [26]:
from IPython.display import display, Markdown
display(Markdown(f'**Score**: {metric.score}'))
display(Markdown(f'**Reason**: {metric.reason}'))

**Score**: 1.0

**Reason**: The score is 1.00 because the output directly addresses the request for a summary of the article, including key characters and plot points, without any irrelevant statements. This high score reflects the complete alignment with the input requirements.

In [27]:
instructions = "You are a scholarly research assistant specialising in technology and organisational studies.Your task is to produce a structured summary of a provided article."
PROMPT = """
    Summarize the following article in at most four paragraphs. Please include all key characters and plot points.
    <article>
    {article}
    </article>
    What is the article all about and who is the target audience
"""

In [28]:
response = client.responses.create(
    model="gpt-4o-mini",
    instructions=instructions,
    input=[
        {"role": "user", 
         "content": PROMPT.format(article=document_text)}
    ],
    temperature=0.7
)

In [29]:
response.output_text

'The article, "The GenAI Divide: State of AI in Business 2025," authored by a team from MIT\'s Project NANDA, explores the current landscape of Generative AI (GenAI) adoption and its impact on businesses. Despite significant investments—estimated between $30-40 billion—only 5% of organizations see a tangible return from their AI initiatives, indicating a stark divide between those who successfully leverage AI and those who do not. The findings are derived from a comprehensive study that included over 300 public AI initiatives, interviews with representatives from 52 organizations, and survey responses from 153 senior leaders.\n\nKey findings reveal that while enterprises are piloting GenAI tools, these often fail to transition into scalable solutions. The article highlights four critical patterns contributing to the GenAI Divide: limited disruption across most sectors, an enterprise paradox where large organizations pilot extensively but struggle to scale, a bias in investment toward v

In [30]:
from deepeval.metrics import SummarizationMetric    

summarization_metric = SummarizationMetric(
    threshold=0.5,
    model=model,  # use GPTModel object so it routes through the API gateway
    assessment_questions=[
        "Does the summary mention the AI adoption gap between large and small companies?",
        "Does the summary cover key statistics from the report?",
        "Does the summary mention challenges faced by businesses adopting AI?",
        "Does the summary include the report's recommendations?",
        "Does the summary reflect the overall findings accurately?"
    ]
)

In [31]:
from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCaseParams

coherence_metric = GEval(
    name="Coherence",
    criteria="Evaluate whether the summary is logically structured and easy to follow.",
    evaluation_steps=[
        "Does the summary have a clear opening?",
        "Do ideas flow logically from one to the next?",
        "Are sentences well-constructed and unambiguous?",
        "Is the summary free of contradictions?",
        "Does the summary have a clear concluding point?"
    ],
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    model=model,
)

tonality_metric = GEval(
    name="Tonality",
    criteria="Evaluate whether the summary consistently uses Formal Academic Writing tone.",
    evaluation_steps=[
        "Does the summary use third-person voice throughout?",
        "Is discipline-appropriate vocabulary used?",
        "Are claims appropriately hedged where needed?",
        "Is colloquial or casual language avoided?",
        "Are sentence structures complex but clear?"
    ],
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    model=model,
)

safety_metric = GEval(
    name="Safety",
    criteria="Evaluate whether the summary is free of harmful or misleading content.",
    evaluation_steps=[
        "Does the summary avoid factual misrepresentations?",
        "Is the summary free of biased or discriminatory language?",
        "Does the summary avoid sensationalism?",
        "Are no false claims presented as facts?",
        "Is the content appropriate for a professional audience?"
    ],
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    model=model,
)

In [32]:
# Build the test case using the structured summary produced above
test_case_eval = LLMTestCase(
    input=user_prompt,
    actual_output=summary_obj.Summary,
    context=[context]  # the truncated document text used for generation
)

# Run all four metrics
summarization_metric.measure(test_case_eval)
coherence_metric.measure(test_case_eval)
tonality_metric.measure(test_case_eval)
safety_metric.measure(test_case_eval)

# Collect results into structured output
class EvaluationResult(BaseModel):
    SummarizationScore: float
    SummarizationReason: str
    CoherenceScore: float
    CoherenceReason: str
    TonalityScore: float
    TonalityReason: str
    SafetyScore: float
    SafetyReason: str

eval_result = EvaluationResult(
    SummarizationScore=summarization_metric.score,
    SummarizationReason=summarization_metric.reason,
    CoherenceScore=coherence_metric.score,
    CoherenceReason=coherence_metric.reason,
    TonalityScore=tonality_metric.score,
    TonalityReason=tonality_metric.reason,
    SafetyScore=safety_metric.score,
    SafetyReason=safety_metric.reason,
)

# Display results
from IPython.display import display, Markdown
display(Markdown(f"""
## Evaluation Results

| Metric | Score | Reason |
|--------|-------|--------|
| Summarization | {eval_result.SummarizationScore:.2f} | {eval_result.SummarizationReason} |
| Coherence | {eval_result.CoherenceScore:.2f} | {eval_result.CoherenceReason} |
| Tonality | {eval_result.TonalityScore:.2f} | {eval_result.TonalityReason} |
| Safety | {eval_result.SafetyScore:.2f} | {eval_result.SafetyReason} |
"""))


## Evaluation Results

| Metric | Score | Reason |
|--------|-------|--------|
| Summarization | 0.83 | The score is 0.83 because the summary effectively captures the main ideas of the original text, but it introduces extra information regarding specific dollar amounts and adoption reluctance that were not present in the original text. This additional information could lead to misunderstandings about the original context. |
| Coherence | 0.86 | The summary has a clear opening that introduces the report's title and main findings. Ideas flow logically, detailing the divide in AI implementation and the challenges faced by organizations. Sentences are well-constructed and unambiguous, effectively conveying complex information. There are no contradictions present, and the summary concludes with a clear point about the need for organizations to address the GenAI Divide. However, a slightly more explicit concluding statement could enhance clarity. |
| Tonality | 0.91 | The summary effectively uses third-person voice throughout and employs discipline-appropriate vocabulary, demonstrating a strong command of the subject matter. Claims are hedged appropriately, particularly regarding the complexities of AI adoption and the varying success rates among organizations. Colloquial language is avoided, maintaining a formal tone. The sentence structures are complex yet clear, facilitating comprehension of intricate ideas. However, a slight improvement could be made in further clarifying the implications of the findings for specific stakeholders. |
| Safety | 0.90 | The summary accurately presents findings from the report without factual misrepresentations and avoids biased or discriminatory language. It does not sensationalize the information and refrains from making false claims. The content is appropriate for a professional audience, providing a clear analysis of the challenges and opportunities related to Generative AI in business. However, it could benefit from a slightly more concise presentation to enhance clarity. |


# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

In [33]:
feedback_notes = []
if eval_result.SummarizationScore < 0.8:
    feedback_notes.append(f"Summarization feedback: {eval_result.SummarizationReason}")
if eval_result.CoherenceScore < 0.8:
    feedback_notes.append(f"Coherence feedback: {eval_result.CoherenceReason}")
if eval_result.TonalityScore < 0.8:
    feedback_notes.append(f"Tonality feedback: {eval_result.TonalityReason}")
if eval_result.SafetyScore < 0.8:
    feedback_notes.append(f"Safety feedback: {eval_result.SafetyReason}")

feedback_block = "\n".join(feedback_notes) if feedback_notes else "All metrics scored well. Minor refinements only."

# Enhanced system prompt incorporating evaluation feedback
enhanced_system_prompt = f"""You are a scholarly research assistant specialising in technology and organisational studies.
Your task is to produce an improved structured summary of a provided article.

A previous version of this summary received the following evaluation feedback:
{feedback_block}

Requirements:
- Address all feedback points explicitly in your revised summary.
- Write in Formal Academic Writing: third-person voice, precise vocabulary, hedged claims, complex but clear sentences.
- The Summary field must be concise and no longer than 1000 tokens.
- The Relevance field must be a single paragraph explaining relevance to AI professionals.
- Populate Author and Title from the document; write 'Not specified' if unknown.
- Set Tone to the exact style name provided by the user.
- Do NOT include token counts — those will be injected programmatically.
- Respond ONLY with structured JSON matching the schema."""

enhanced_user_prompt = f"""Please produce an improved summary of the following article using the tone: {selected_tone}.

Article content:
{context}
"""

# Re-generate with enhanced prompt
enhanced_response = client.beta.chat.completions.parse(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": enhanced_system_prompt},
        {"role": "user",   "content": enhanced_user_prompt},
    ],
    response_format=ArticleSummary,
)

enhanced_summary = enhanced_response.choices[0].message.parsed
enhanced_summary.InputTokens  = enhanced_response.usage.prompt_tokens
enhanced_summary.OutputTokens = enhanced_response.usage.completion_tokens

# Re-evaluate using the same metrics
enhanced_test_case = LLMTestCase(
    input=enhanced_user_prompt,
    actual_output=enhanced_summary.Summary,
    context=[context]
)

summarization_metric.measure(enhanced_test_case)
coherence_metric.measure(enhanced_test_case)
tonality_metric.measure(enhanced_test_case)
safety_metric.measure(enhanced_test_case)

enhanced_eval = EvaluationResult(
    SummarizationScore=summarization_metric.score,
    SummarizationReason=summarization_metric.reason,
    CoherenceScore=coherence_metric.score,
    CoherenceReason=coherence_metric.reason,
    TonalityScore=tonality_metric.score,
    TonalityReason=tonality_metric.reason,
    SafetyScore=safety_metric.score,
    SafetyReason=safety_metric.reason,
)

# Compare original vs enhanced scores
display(Markdown(f"""
## Enhanced Summary

{enhanced_summary.Summary}

## Score Comparison: Original vs Enhanced

| Metric | Original | Enhanced |
|--------|----------|----------|
| Summarization | {eval_result.SummarizationScore:.2f} | {enhanced_eval.SummarizationScore:.2f} |
| Coherence | {eval_result.CoherenceScore:.2f} | {enhanced_eval.CoherenceScore:.2f} |
| Tonality | {eval_result.TonalityScore:.2f} | {enhanced_eval.TonalityScore:.2f} |
| Safety | {eval_result.SafetyScore:.2f} | {enhanced_eval.SafetyScore:.2f} |
"""))


## Enhanced Summary

The report "The GenAI Divide: State of AI in Business 2025" provides a comprehensive analysis of contemporary generative AI (GenAI) implementations across various industries, revealing that despite substantial investments amounting to $30–40 billion, approximately 95% of organizations fail to achieve any significant return on their GenAI initiatives. This phenomenon, termed the GenAI Divide, illustrates a paradox wherein 5% of pilot projects yield considerable value, while the majority do not impact profit-and-loss statements. Key findings indicate that widespread adoption of tools such as ChatGPT and Copilot does not necessarily lead to transformative outcomes, as these tools primarily enhance individual productivity rather than drive organizational change. The research identifies critical barriers that hinder successful implementation, including brittle workflows, insufficient contextual learning, and a misalignment of GenAI tools with existing operational practices. Four central patterns emerge: 1) limited disruption across most sectors, with only two exhibiting significant structural changes; 2) large enterprises leading in pilot initiatives yet lagging in scaling; 3) an investment bias favoring visible ROI over long-term strategic benefits; and 4) increased success rates for organizations that engage in external partnerships for AI development. These insights underscore that the primary obstruction to scaling GenAI is not infrastructural or regulatory but rather a deficit in adaptive learning. Organizations that effectively navigate the divide emphasize customization and alignment with business objectives, leading to accelerated deployment and measurable savings even absent comprehensive restructuring. Overall, this report confirms that while numerous enterprises engage with GenAI technologies, few manage to transition from pilot projects to impactful implementations, thereby reinforcing the notion that mere adoption does not equate to meaningful transformation.

## Score Comparison: Original vs Enhanced

| Metric | Original | Enhanced |
|--------|----------|----------|
| Summarization | 0.83 | 0.67 |
| Coherence | 0.86 | 0.87 |
| Tonality | 0.91 | 0.90 |
| Safety | 0.90 | 0.91 |


Please, do not forget to add your comments.


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
